In [ ]:
#importa a biblioteca SQLite3, que é nativa de Python
import sqlite3 
#mostra o caminho onde está o Banco de Dados
db_path = '../data/portfolio.db' 
#Usa o método .connect para conectar a biblioteca sqlite3 com o banco de dados
conn = sqlite3.connect(db_path)
# O método .cursor() é o responsável por fazer com que Pytohn entenda os comandos SQL
cursor = conn.cursor()
print("Banco de dados pronto para receber comandos!")

Banco de dados pronto para receber comandos!


In [ ]:
#importar a biblioteca yahoo finance e ver a estrutura dos dados
import yfinance as yf

# Vamos "espiar" os dados da Apple
teste = yf.download("AAPL", period="5d")
print(teste.head())

Criação das tabelas . Usa o conn.commit

In [5]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_assets(
    asset_id INTEGER PRIMARY KEY AUTOINCREMENT,
    ticker TEXT NOT NULL UNIQUE,
    asset_name TEXT,
    asset_type TEXT
)
""") 
conn.commit()

In [11]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_prices(
        price_id INTEGER PRIMARY KEY AUTOINCREMENT,
        asset_id INTEGER NOT NULL,
        date DATE,
        close_price REAL,
        volume INTEGER,
FOREIGN KEY (asset_id) REFERENCES dim_assets (asset_id))
""")
conn.commit()

In [15]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_portfolios(
    portfolio_id INTEGER PRIMARY KEY AUTOINCREMENT,
    risk_profile TEXT,
    asset_id INTEGER, 
    weight REAL,
    FOREIGN KEY (asset_id) REFERENCES dim_assets (asset_id))   
""")
conn.commit()

In [1]:
import sqlite3
import pandas as pd

# 1. Conectar ao banco (lembre-se do '../' porque o notebook está na pasta /notebooks)
conn = sqlite3.connect('../data/portfolio.db')

# 2. Usar o Pandas para ler a tabela
# O comando SQL "SELECT * FROM nome_da_tabela" significa: "Selecione TUDO da tabela"
query = "SELECT * FROM dim_assets"
df_ativos = pd.read_sql_query(query, conn)

# 3. Fechar a conexão
conn.close()

# 4. Exibir a tabela
df_ativos

,asset_id,ticker,asset_name,asset_type
0,1,PETR4.SA,Petrobrás,Ação BR
1,2,LFTS11.SA,Tesouro Selic ETF,Renda Fixa
2,3,BTLG11.SA,BTG Logística,FII
3,4,AAPL,Apple Inc,Stock US
4,5,GOLD11.SA,Ouro ETF,Commodity
5,6,BTC-USD,Bitcoin,Cripto


In [ ]:
import yfinance as yf
import sqlite3
import pandas as pd

# 1. Buscar os tickers que você cadastrou no banco
conn = sqlite3.connect('../data/portfolio.db')
df_assets = pd.read_sql_query("SELECT asset_id, ticker FROM dim_assets", conn)
conn.close()

# 2. Criar uma lista só com os tickers (ex: ['PETR4.SA', 'AAPL', ...])
tickers_list = df_assets['ticker'].tolist()

# 3. Baixar dados da API (vamos pegar os últimos 6 meses para testar)
print(f"Baixando dados para: {tickers_list}")
dados_brutos = yf.download(tickers_list, period="6mo")["Close"]

# 4. Ver como os dados chegaram
dados_brutos.head()

Baixando dados para: ['AAPL', 'BTC-USD', 'BTLG11.SA', 'GOLD11.SA', 'LFTS11.SA', 'PETR4.SA']


[*********************100%***********************]  6 of 6 completed


Ticker,AAPL,BTC-USD,BTLG11.SA,GOLD11.SA,LFTS11.SA,PETR4.SA
Date,,,,,,
2025-08-20,NaN,114274.742188,93.509056,19.139999,138.289993,28.644966
2025-08-21,224.472153,112419.031250,94.530754,19.110001,138.399994,28.711319
2025-08-22,227.326706,116874.085938,94.301598,19.110001,138.460007,29.537159
2025-08-23,NaN,115374.328125,NaN,NaN,NaN,NaN
2025-08-24,NaN,113458.429688,NaN,NaN,NaN,NaN
